In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations, permutations
from matplotlib.markers import MarkerStyle
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
import matplotlib.lines as mlines
from matplotlib.patches import Wedge

In [ ]:
#initialize a datacube to store results
reg = ['BI', 'IP', 'FR', 'ME', 'AL', 'SEA', 'NEA', 'SC', 'WMD', 'EMD']
season = ['all_seasons', 'MAM', 'JJA', 'SON', 'DJF']

In [ ]:
neve = xr.Dataset(
        data_vars = {
        'mean_event_number': (('x', 'y', 'time'), 
                  np.full((len(reg), len(reg), len(season)), np.nan)),
        'std_event_number': (('x', 'y', 'time'), 
                  np.full((len(reg), len(reg), len(season)), np.nan))
        },
        coords = {
            'x': reg,
            'y': reg,
            'time': season
        }
)

In [ ]:
neve

In [ ]:
for s in season:

    df = pd.read_csv(f'/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair/Synchronized_event_pair_between_region_pair_{s}.csv')
    
    for reg1, reg2 in combinations(reg, 2):
        
        df_region_pair = df[(df['reg1'] == reg1) & (df['reg2'] == reg2)]
        
        neve['mean_event_number'].loc[dict(x = reg1, y = reg2, time = s)] = df_region_pair['num2'].mean()
        neve['std_event_number'].loc[dict(x = reg1, y = reg2, time = s)] = df_region_pair['num2'].std()

        neve['mean_event_number'].loc[dict(x = reg2, y = reg1, time = s)] = df_region_pair['num1'].mean()
        neve['std_event_number'].loc[dict(x = reg2, y = reg1, time = s)] = df_region_pair['num1'].std()

In [ ]:
#save
#neve.to_netcdf('/net/rain/hyclimm/data/projects/SynFire/WP1/syn_event_number_region_pair/Average_event_number_during_synchronicity.nc')

In [ ]:
#plot
bounds = [1, 2, 3, 4, 5, 6, 7]
cmap = ListedColormap(['#fde725ff', 
                       '#7ad151ff',
                       '#22a884ff',
                       '#2a788eff',
                       '#414487ff',
                       '#440154ff'])
norm = BoundaryNorm(bounds, cmap.N)

In [ ]:
cmap

In [ ]:
fig, axes = plt.subplots(2, 3, figsize = (15, 10))
axes = axes.flatten()
reg_ord = ['IP', 'WMD', 'EMD', 'FR', 'AL', 'SEA', 'BI', 'ME', 'NEA', 'SC']
reg_lb = ['IP', 'WMD', 'EMD', 'FR', 'AL', 'SEE', 'BI', 'CE', 'NEE', 'SC'] #N-S, name changed

for i, ax in enumerate(axes):

    if i == 5:
        ax.axis('off')
    else:
        # global axes setting
        ax.set_xlim(-0.5, 9.5)
        ax.set_ylim(-0.5, 9.5)
        ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], reg_lb)
        ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], reg_lb)
        ax.tick_params(axis = 'both',
                       labelsize = 10)
        ax.set_aspect('equal')
        
        val = neve['mean_event_number'].sel(time = season[i]).copy()

        #all seasons ES < 10 not shown, individual season ES < 5 not shown
        ES = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_Sig_Sns/All_Fires/Snstest/ES_matrix_taumax_6_{season[i]}.csv", index_col = 0)

        
        #aggregate the matrix to upper right
        for m in range(len(ES)):
            for n in range(m, len(ES)):
                if m == n:
                    ES.iloc[m,n] = 0
                else:
                    ES.iloc[m,n] += ES.iloc[n,m]
                    ES.iloc[n,m] = 0
    
        for reg1, reg2 in combinations(reg_ord, 2):
    
            x_pos = reg_ord.index(reg1)
            y_pos = reg_ord.index(reg2)
    
            v1 = val.sel(x = reg1, y = reg2).values
            v2 = val.sel(x = reg2, y = reg1).values
            
            if np.isnan(v1) or np.isnan(v2):
                continue

            radius = 0.4
            
            w1 = Wedge((x_pos, y_pos), radius, 45, 225, 
                       facecolor = cmap(norm(v1)),
                       edgecolor = 'black',
                       linewidth = 0.8)
            
            w2 = Wedge((x_pos, y_pos), radius, 225, 405, 
                       facecolor = cmap(norm(v2)),
                       edgecolor='black',
                       linewidth = 0.8)
            
            ax.add_patch(w1)
            ax.add_patch(w2)
            
            

axes[0].set_title('(a) All seasons', fontsize = 15)
axes[1].set_title('(b) MAM', fontsize = 15)
axes[2].set_title('(c) JJA', fontsize = 15)
axes[3].set_title('(d) SON', fontsize = 15)
axes[4].set_title('(e) DJF', fontsize = 15)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # no actual data needed

# Add the colorbar
cax = fig.add_axes([0.68, 0.35, 0.2, 0.015])  # left botton width height
cbar = plt.colorbar(sm, cax = cax, orientation='horizontal', extend='max')
cbar.set_label('Average event number', fontsize = 12)
cbar.ax.set_xticklabels([1, 2, 3, 4, 5, 6, 7], fontsize = 12)
cbar.ax.xaxis.set_label_position('top')

# Add legend
y = Wedge((1.4, 0.4), 0.05, 45, 225, 
          facecolor='white',
          edgecolor = 'black',
          linewidth = 0.8,
          transform = axes[4].transAxes,
          clip_on = False)

x = Wedge((1.4, 0.25), 0.05, 225, 405, 
          facecolor = 'white',
          edgecolor = 'black',
          linewidth = 0.8,
          transform = axes[4].transAxes,
          clip_on = False)


axes[4].add_artist(y)
axes[4].add_artist(x)
axes[4].text(1.5, 0.4,  'Region on the y-axis', fontsize = 12,
             transform = axes[4].transAxes)
axes[4].text(1.5, 0.25, 'Region on the x-axis', fontsize = 12,
             transform = axes[4].transAxes)


plt.savefig('/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_average_event_number_during_synchronicity.png', dpi = 600, bbox_inches = 'tight')